# Facial Expression Classification — 6CS012 Final Portfolio
## Part II: Image Classification with Convolutional Neural Networks

**Herald College Kathmandu | University of Wolverhampton**

This notebook implements an end-to-end deep learning pipeline for facial expression recognition across seven emotion classes using:
- **Part A**: Baseline CNN (3 Conv + 3 Dense) and Deeper CNN with regularisation
- **Part B**: Transfer Learning with MobileNetV2
- **Experiments**: SGD vs Adam optimiser comparison and Ablation Study
- **Extra**: Grad-CAM visualisations and interactive GUI

## Cell 1 — Setup: Mount Drive & Install Libraries

In [2]:
from pathlib import Path
import os

def _is_dataset_root(path: Path) -> bool:
    # Flexible check: must have 'train' and 'test' subfolders.
    p = str(path)
    has_train = os.path.isdir(os.path.join(p, 'train'))
    has_test  = os.path.isdir(os.path.join(p, 'test'))
    return has_train and has_test

def resolve_project_root():
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        for p in kaggle_input.rglob('train'):
            if p.is_dir() and (p.parent / 'test').is_dir():
                return p.parent.resolve()

    candidates = []
    env_root = os.environ.get('FER_BASE_DIR', '').strip()
    if env_root: candidates.append(Path(env_root))

    # Priority paths for local/Colab
    candidates.extend([
        Path(r'D:\Herald college Kathmandu\AI\Main Assignment\Maain\facial expression classification'),
        Path.cwd(),
        Path('/content/drive/MyDrive/AiMain/facial expression classification'),
        Path('/content/facial expression classification'),
    ])

    seen = set()
    unique_candidates = []
    for c in candidates:
        if str(c) not in seen:
            seen.add(str(c))
            unique_candidates.append(c)

    for base in unique_candidates:
        if _is_dataset_root(base):
            return base.resolve()

    searched = '\n'.join(f'  - {p}' for p in unique_candidates)
    raise FileNotFoundError(f'Could not locate dataset root. Searched:\n{searched}')

BASE_DIR = resolve_project_root()
TRAIN_DIR = BASE_DIR / 'train'
VAL_DIR   = BASE_DIR / 'validation'
TEST_DIR  = BASE_DIR / 'test'


In [3]:
print("TRAIN DIR:", TRAIN_DIR)
print("EXISTS:", TRAIN_DIR.exists())

TRAIN DIR: /kaggle/input/datasets/dipeshmahato/fer2013-custom-3-split/train
EXISTS: True


In [4]:
!python -m pip install --upgrade pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


## Cell 2 — Imports & Reproducibility

In [19]:
# ==============================
# BASIC SYSTEM IMPORTS
# ==============================
import warnings
import os
import sys
import io
import time
import json
import platform

warnings.filterwarnings('ignore')


# ==============================
# NUMPY / PANDAS / VISUALIZATION
# ==============================
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import seaborn as sns
sns.set(style="whitegrid")


# ==============================
# FILE & IMAGE HANDLING
# ==============================
from pathlib import Path
from PIL import Image
from collections import Counter

import cv2

# Pillow compatibility fix
if hasattr(Image, 'Resampling'):
    BILINEAR = Image.Resampling.BILINEAR
else:
    BILINEAR = Image.BILINEAR


# ==============================
# TENSORFLOW / DEEP LEARNING
# ==============================
import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, Conv2D, MaxPooling2D,
    BatchNormalization, GlobalAveragePooling2D,
    Input, Flatten
)

from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

from tensorflow.keras.preprocessing.image import ImageDataGenerator


# ==============================
# SKLEARN METRICS & UTILITIES
# ==============================
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.isotonic import IsotonicRegression



try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output
    WIDGETS_AVAILABLE = True
except:
    WIDGETS_AVAILABLE = False


# ==============================
# REPRODUCIBILITY SETTINGS
# ==============================
np.random.seed(42)
tf.random.set_seed(42)


# ==============================
# SYSTEM INFO OUTPUT
# ==============================
print("=" * 60)
print("  FACIAL EXPRESSION CLASSIFICATION — 6CS012")
print("=" * 60)

print(f"  TensorFlow : {tf.__version__}")
print(f"  Python     : {sys.version.split()[0]}")

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"  Hardware   : GPU detected ({len(gpus)} GPU(s))")
    for g in gpus:
        print(f"               {g}")
else:
    print("  Hardware   : CPU only — enable GPU for faster training")

print("  All imports successful!")

  FACIAL EXPRESSION CLASSIFICATION — 6CS012
  TensorFlow : 2.19.0
  Python     : 3.12.12
  Hardware   : GPU detected (1 GPU(s))
               PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
  All imports successful!


## Cell 3 — Configuration

In [6]:
# ─── Dataset paths (auto-resolved from Cell 1) ─────────────────────────────
print('Dataset path check:')
for name, p in [('TRAIN', TRAIN_DIR), ('VAL', VAL_DIR), ('TEST', TEST_DIR)]:
    status = 'EXISTS' if p.exists() else 'MISSING'
    print(f'  {name}: {p}  [{status}]')

missing = [str(p) for p in (TRAIN_DIR, VAL_DIR, TEST_DIR) if not p.exists()]
if missing:
    raise FileNotFoundError('Missing dataset folders:\n' + '\n'.join(missing))

# ─── Image settings ────────────────────────────────────────────────────────
IMG_SIZE    = 48   # Grayscale CNN input
IMG_SIZE_TL = 96   # MobileNetV2 RGB input
CHANNELS    = 1
CHANNELS_TL = 3

# ─── Training settings ─────────────────────────────────────────────────────
BATCH_SIZE  = 32
EPOCHS      = 50
EPOCHS_FINE = 50
ANGRY_BOOST = 1.35 

# ─── Output folder ─────────────────────────────────────────────────────────
REPORT_DIR = Path.cwd() / 'report_artifacts'
REPORT_DIR.mkdir(exist_ok=True)

np.random.seed(42)
tf.random.set_seed(42)

print('\nConfiguration loaded:')
print(f'  BASE_DIR    : {BASE_DIR}')
print(f'  IMG_SIZE    : {IMG_SIZE}  |  IMG_SIZE_TL : {IMG_SIZE_TL}')
print(f'  BATCH_SIZE  : {BATCH_SIZE}  |  EPOCHS : {EPOCHS}')
print(f'  REPORT_DIR  : {REPORT_DIR}')



Dataset path check:
  TRAIN: /kaggle/input/datasets/dipeshmahato/fer2013-custom-3-split/train  [EXISTS]
  VAL: /kaggle/input/datasets/dipeshmahato/fer2013-custom-3-split/validation  [EXISTS]
  TEST: /kaggle/input/datasets/dipeshmahato/fer2013-custom-3-split/test  [EXISTS]

Configuration loaded:
  BASE_DIR    : /kaggle/input/datasets/dipeshmahato/fer2013-custom-3-split
  IMG_SIZE    : 48  |  IMG_SIZE_TL : 96
  BATCH_SIZE  : 32  |  EPOCHS : 50
  REPORT_DIR  : /kaggle/working/report_artifacts


## Cell 4 — Load Dataset

The FER dataset is pre-organised into `train`, `validation`, and `test` directories, each containing sub-folders per emotion class.

**Split Justification:** The dataset arrives pre-split, which we preserve without modification to eliminate data leakage. The training set fits model weights, validation is used for early stopping and hyperparameter tuning, and the test set is reserved exclusively for final evaluation. This three-way split ensures reported test metrics reflect genuine generalisation performance.

In [7]:
print('=' * 60)
print('  CELL 4 — LOAD DATASET')
print('=' * 60)

CLASS_FOLDERS = sorted([d.name for d in TRAIN_DIR.iterdir() if d.is_dir()])
EMOTION_LABELS = [c.capitalize() for c in CLASS_FOLDERS]
N_CLASSES = len(EMOTION_LABELS)
label_to_idx = {folder: i for i, folder in enumerate(CLASS_FOLDERS)}
print(f'  Classes ({N_CLASSES}): {EMOTION_LABELS}')

def load_split(split_dir, img_size=IMG_SIZE):
    images, labels = [], []
    for folder in CLASS_FOLDERS:
        cls_dir = split_dir / folder
        if not cls_dir.exists():
            continue
        idx = label_to_idx[folder]
        for img_path in cls_dir.iterdir():
            if img_path.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                continue
            try:
                img = Image.open(img_path).convert('L')
                img = img.resize((img_size, img_size), BILINEAR)
                arr = np.array(img, dtype=np.float32) / 255.0
                images.append(arr[..., np.newaxis])
                labels.append(idx)
            except Exception:
                continue
    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32)

print('\nLoading train split …')
X_train, y_train = load_split(TRAIN_DIR)
print(f'  X_train: {X_train.shape}  y_train: {y_train.shape}')

print('Loading validation split …')
X_val, y_val = load_split(VAL_DIR)
print(f'  X_val  : {X_val.shape}   y_val  : {y_val.shape}')

print('Loading test split …')
X_test, y_test = load_split(TEST_DIR)
print(f'  X_test : {X_test.shape}  y_test : {y_test.shape}')

if min(len(X_train), len(X_val), len(X_test)) == 0:
    raise RuntimeError('One or more splits are empty. Check train/validation/test paths and folder contents.')

# Save dataset info
angry_index = next((i for i, lbl in enumerate(EMOTION_LABELS) if lbl.lower() == 'angry'), 0)

info = {
    'train_count': int(len(X_train)),
    'val_count':   int(len(X_val)),
    'test_count':  int(len(X_test)),
    'classes':     EMOTION_LABELS,
    'class_folders': CLASS_FOLDERS,
}
with open(REPORT_DIR / 'dataset_info.json', 'w') as f:
    json.dump(info, f, indent=2)
print('\nSaved: dataset_info.json')

# Save env info
env = {
    'python':     platform.python_version(),
    'platform':   platform.platform(),
    'tensorflow': tf.__version__,
    'numpy':      np.__version__,
    'opencv':     cv2.__version__,
}
with open(REPORT_DIR / 'env_info.json', 'w') as f:
    json.dump(env, f, indent=2)
print('Saved: env_info.json')
print('\nDataset loaded successfully!')

  CELL 4 — LOAD DATASET
  Classes (7): ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

Loading train split …
  X_train: (26872, 48, 48, 1)  y_train: (26872,)
Loading validation split …
  X_val  : (6866, 48, 48, 1)   y_val  : (6866,)
Loading test split …
  X_test : (1670, 48, 48, 1)  y_test : (1670,)

Saved: dataset_info.json
Saved: env_info.json

Dataset loaded successfully!


## Cell 5 — Face-Crop Preprocessing Pipeline

An OpenCV Haar Cascade face detector is applied to all images, cropping the facial region with 20% contextual padding. This preprocessing is applied **identically** at training and inference time to maintain consistency and ensure the model is not exposed to background noise during training.

In [8]:
print("  CELL 5 — FACE-CROP PREPROCESSING PIPELINE")

# Load Haar Cascade face detector
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

assert not face_cascade.empty(), "Failed to load Haar Cascade!"
print("  Face detector loaded.")


def _largest_face_box(gray_img):
    """
    Detect faces and return the largest face box as:
    (x, y, w, h)

    Returns:
        None if no face is detected.
    """

    faces = face_cascade.detectMultiScale(
        gray_img,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    if len(faces) == 0:
        return None

    largest = max(faces, key=lambda b: b[2] * b[3])

    return largest


def crop_face_from_array(img_array):
    """
    Crop the largest detected face from a (H,W,1) float32 array.
    Falls back to original image if no face detected.
    """

    if img_array.ndim == 3 and img_array.shape[-1] == 1:
        pil_img = Image.fromarray(
            (img_array.squeeze() * 255).astype('uint8'),
            mode='L'
        )
    else:
        arr = np.clip(img_array * 255.0, 0, 255).astype('uint8')
        pil_img = Image.fromarray(arr)

    rgb = np.array(pil_img.convert('RGB'))

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)

    face = _largest_face_box(gray)

    if face is None:
        return pil_img.convert('RGB')

    x, y, w, h = face

    pad = int(0.20 * max(w, h))

    x1 = max(0, x - pad)
    y1 = max(0, y - pad)

    x2 = min(rgb.shape[1], x + w + pad)
    y2 = min(rgb.shape[0], y + h + pad)

    cropped = rgb[y1:y2, x1:x2]

    return Image.fromarray(cropped)


def build_cropped_gray(X_arr, desc=""):

    out = []

    for i, img in enumerate(X_arr):

        pil = crop_face_from_array(img).convert('L')

        pil = pil.resize((IMG_SIZE, IMG_SIZE), BILINEAR)

        arr = np.array(pil, dtype=np.float32) / 255.0

        out.append(arr[..., np.newaxis])

        if (i + 1) % 5000 == 0:
            print(f"    {desc}: {i+1:,}/{len(X_arr):,}")

    return np.stack(out, dtype=np.float32)


def build_cropped_rgb(X_arr, desc=""):

    out = []

    for i, img in enumerate(X_arr):

        pil = crop_face_from_array(img).convert('RGB')

        pil = pil.resize((IMG_SIZE_TL, IMG_SIZE_TL), BILINEAR)

        arr = np.array(pil, dtype=np.float32)

        arr = mobilenet_preprocess(arr)

        out.append(arr)

        if (i + 1) % 5000 == 0:
            print(f"    {desc}: {i+1:,}/{len(X_arr):,}")

    return np.stack(out, dtype=np.float32)


print("\nBuilding face-cropped datasets (this may take several minutes) …")

X_train_orig = X_train.copy()
X_val_orig   = X_val.copy()
X_test_orig  = X_test.copy()

X_train = build_cropped_gray(X_train, "train gray")
X_val   = build_cropped_gray(X_val,   "val gray")
X_test  = build_cropped_gray(X_test,  "test gray")

X_tr_rgb   = build_cropped_rgb(X_train_orig, "train RGB")
X_val_rgb  = build_cropped_rgb(X_val_orig,   "val RGB")
X_test_rgb = build_cropped_rgb(X_test_orig,  "test RGB")

print("\nFinal dataset shapes:")

print(
    f"  Grayscale  — "
    f"train: {X_train.shape}  "
    f"val: {X_val.shape}  "
    f"test: {X_test.shape}"
)

print(
    f"  RGB        — "
    f"train: {X_tr_rgb.shape}  "
    f"val: {X_val_rgb.shape}  "
    f"test: {X_test_rgb.shape}"
)

print(
    f"  Data range — "
    f"gray [{X_train.min():.2f}, {X_train.max():.2f}]  "
    f"rgb [{X_tr_rgb.min():.2f}, {X_tr_rgb.max():.2f}]"
)

print("\nPREPROCESSING PARITY:")
print("  Train / Val / Test / GUI inference — all use IDENTICAL face-crop pipeline")
print("  SUCCESS: Preprocessing consistent across all splits and inference.")
import gc; del X_train_orig, X_val_orig, X_test_orig; gc.collect()

  CELL 5 — FACE-CROP PREPROCESSING PIPELINE
  Face detector loaded.

Building face-cropped datasets (this may take several minutes) …
    train gray: 5,000/26,872
    train gray: 10,000/26,872
    train gray: 15,000/26,872
    train gray: 20,000/26,872
    train gray: 25,000/26,872
    val gray: 5,000/6,866
    train RGB: 5,000/26,872
    train RGB: 10,000/26,872
    train RGB: 15,000/26,872
    train RGB: 20,000/26,872
    train RGB: 25,000/26,872
    val RGB: 5,000/6,866

Final dataset shapes:
  Grayscale  — train: (26872, 48, 48, 1)  val: (6866, 48, 48, 1)  test: (1670, 48, 48, 1)
  RGB        — train: (26872, 96, 96, 3)  val: (6866, 96, 96, 3)  test: (1670, 96, 96, 3)
  Data range — gray [0.00, 1.00]  rgb [-1.00, 1.00]

PREPROCESSING PARITY:
  Train / Val / Test / GUI inference — all use IDENTICAL face-crop pipeline
  SUCCESS: Preprocessing consistent across all splits and inference.


0

## Cell 6 — Data Understanding, Analysis & Visualisation

In [9]:
print("=" * 60)
print("  CELL 6 — DATA UNDERSTANDING, ANALYSIS & VISUALISATION")
print("=" * 60)

# ── 6.1 Class distribution ───────────────────────────────────────────────────
train_counts = Counter(y_train.tolist())
test_counts  = Counter(y_test.tolist())

print("\n6.1  Class Distribution")
print(f"{'Emotion':<12} {'Train':>8}  {'Test':>7}  {'Train%':>7}")
print("-" * 40)
for i, lbl in enumerate(EMOTION_LABELS):
    tr = train_counts.get(i, 0)
    te = test_counts.get(i, 0)
    print(f"  {lbl:<10} {tr:>8,}  {te:>7,}  {tr/len(y_train)*100:>6.1f}%")
print(f"  {'TOTAL':<10} {len(y_train):>8,}  {len(y_test):>7,}  100.0%")

# Bar + Pie chart
cmap = plt.get_cmap("Set2")
colors = [cmap(i) for i in range(N_CLASSES)]
tr_vals = [train_counts.get(i,0) for i in range(N_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
bars = axes[0].bar(EMOTION_LABELS, tr_vals, color=colors, edgecolor='black', linewidth=0.6)
axes[0].set_title('Training Set — Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(EMOTION_LABELS, rotation=30, ha='right')
for b, v in zip(bars, tr_vals):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+20,
                 f'{v:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')

axes[1].pie(tr_vals, labels=EMOTION_LABELS, colors=colors,
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Training Set — Class Share', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'class_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print("  Saved: class_distribution.png")

# ── 6.2 Sample images (5 per class) ──────────────────────────────────────────
print("\n6.2  Sample Images Per Class")
fig, axes = plt.subplots(N_CLASSES, 5, figsize=(12, N_CLASSES * 2))
for row, emotion_idx in enumerate(range(N_CLASSES)):
    idxs = np.where(y_train == emotion_idx)[0][:5]
    for col, idx in enumerate(idxs):
        axes[row, col].imshow(X_train[idx].squeeze(), cmap='gray', vmin=0, vmax=1)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(EMOTION_LABELS[emotion_idx],
                                      fontsize=10, fontweight='bold',
                                      rotation=0, labelpad=60)
plt.suptitle('Sample Images — 5 per Emotion Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'sample_images.png', dpi=200, bbox_inches='tight')
plt.show()
print("  Saved: sample_images.png")

# ── 6.3 Pixel intensity statistics ───────────────────────────────────────────
flat = X_train.flatten()
print(f"\n6.3  Pixel intensity stats — Mean: {flat.mean():.4f}  Std: {flat.std():.4f}  Min: {flat.min():.4f}  Max: {flat.max():.4f}")


  CELL 6 — DATA UNDERSTANDING, ANALYSIS & VISUALISATION

6.1  Class Distribution
Emotion         Train     Test   Train%
----------------------------------------
  Angry         3,686      100    13.7%
  Disgust         329      100     1.2%
  Fear          3,796      270    14.1%
  Happy         6,857      300    25.5%
  Neutral       4,675      300    17.4%
  Sad           4,631      300    17.2%
  Surprise      2,898      300    10.8%
  TOTAL        26,872    1,670  100.0%
  Saved: class_distribution.png

6.2  Sample Images Per Class
  Saved: sample_images.png

6.3  Pixel intensity stats — Mean: 0.5068  Std: 0.2550  Min: 0.0000  Max: 1.0000


## Cell 7 — Class Weights & Data Augmentation

Class weighting and augmentation are tuned for stability and better generalization:
- Class weights are balanced, mildly boosted for Angry, and clipped to avoid unstable gradients
- Augmentation is moderate (rotation, shift, zoom, shear, brightness, horizontal flip)
- Train/validation generators preserve preprocessing consistency (rescale only on validation)

In [30]:
print('  CELL 7 — CLASS WEIGHTS & DATA AUGMENTATION')


# ── 7.1 CLASS WEIGHTS (FIXED + SAFE) ─────────────────────────────
uniq = np.unique(y_train)

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=uniq,
    y=y_train
)

class_weights = {int(c): float(w) for c, w in zip(uniq, class_weights_arr)}

# Optional mild class adjustment (ONLY if justified in viva)
# NOTE: Avoid strong manual bias unless explained
if 'angry_index' in globals():
    class_weights[angry_index] = class_weights[angry_index] * 1.10

# Clip extreme values for stability
class_weights = {
    k: float(np.clip(v, 0.90, 2.20))
    for k, v in class_weights.items()
}

print('  Balanced class weights:')
for i, lbl in enumerate(EMOTION_LABELS):
    print(f'    {lbl:<10}: {class_weights.get(i,1.0):.3f}')


# ── 7.2 DATA AUGMENTATION (FIXED FOR GRAYSCALE) ──────────────────
from tensorflow.keras.preprocessing.image import ImageDataGenerator

aug = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.08,
    fill_mode='nearest'
)

# IMPORTANT FIX:
# Keep data in [0,1] range (no unnecessary *255 scaling)
X_train_aug = X_train.astype('float32')
X_val_aug   = X_val.astype('float32')


# ── VISUALIZATION ────────────────────────────────────────────────
print('\n  Visualising augmentation examples …')

fig, axes = plt.subplots(2, 5, figsize=(14, 6))

sample_idxs = [np.where(y_train == i)[0][0] for i in range(min(5, N_CLASSES))]

for col, idx in enumerate(sample_idxs):

    # Original image
    orig = X_train[idx]
    axes[0, col].imshow(orig.squeeze(), cmap='gray')
    axes[0, col].set_title(f'Original\n{EMOTION_LABELS[y_train[idx]]}', fontsize=8)
    axes[0, col].axis('off')

    # Augmented image
    sample_x = X_train_aug[idx:idx+1]
    sample_y = y_train[idx:idx+1]

    aug_iter = aug.flow(sample_x, sample_y, batch_size=1, shuffle=False)
    aug_img = next(aug_iter)[0]

    axes[1, col].imshow(aug_img.squeeze(), cmap='gray')
    axes[1, col].set_title('Augmented', fontsize=8)
    axes[1, col].axis('off')

plt.suptitle('Data Augmentation Examples', fontsize=13, fontweight='bold')
plt.tight_layout()

plt.savefig(REPORT_DIR / 'augmentation_samples.png', dpi=200, bbox_inches='tight')
plt.show()

print('  Saved: augmentation_samples.png')


# ── GENERATORS ───────────────────────────────────────────────────
train_aug_gen = aug.flow(
    X_train_aug, y_train,
    batch_size=BATCH_SIZE,
    seed=42
)

steps_per_epoch = int(np.ceil(len(X_train_aug) / BATCH_SIZE))

val_gen = ImageDataGenerator().flow(
    X_val.astype('float32'),
    y_val,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_steps = int(np.ceil(len(X_val) / BATCH_SIZE))

print(f'\n  Generator ready — steps/epoch: {steps_per_epoch} | val_steps: {val_steps}')
print('  Augmentation applied: rotation, shift, flip, zoom, shear')

  CELL 7 — CLASS WEIGHTS & DATA AUGMENTATION
  Balanced class weights:
    Angry     : 1.146
    Disgust   : 2.200
    Fear      : 1.011
    Happy     : 0.900
    Neutral   : 0.900
    Sad       : 0.900
    Surprise  : 1.325

  Visualising augmentation examples …
  Saved: augmentation_samples.png

  Generator ready — steps/epoch: 840 | val_steps: 215
  Augmentation applied: rotation, shift, flip, zoom, shear


## Cell 8 — Part A: Baseline CNN (3 Conv + 3 Dense)

Architecture complies exactly with the assignment specification:
- 3 Convolutional layers with increasing filters (32→64→128), each followed by MaxPooling
- 3 Fully Connected layers (Dense 256 → Dense 128 → Output Dense 7 with Softmax)
- Batch Normalization and Dropout for regularization
- Compiled with Adam (lr=3e-4), sparse categorical cross-entropy

In [31]:
def build_baseline_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), n_classes=N_CLASSES):
    """
    Spec-compliant baseline:
      Conv1 → Pool1 | Conv2 → Pool2 | Conv3 → Pool3
      FC1(256) | FC2(128) | Output(n_classes)
    """
    model = Sequential([
        # ── Convolutional Block 1 ────────────────────────────────
        Conv2D(32, (3,3), padding='same', activation='relu',
               input_shape=input_shape, name='conv1'),
        BatchNormalization(name='bn1'),
        MaxPooling2D(name='pool1'),
        Dropout(0.20, name='drop1'),

        # ── Convolutional Block 2 ────────────────────────────────
        Conv2D(64, (3,3), padding='same', activation='relu', name='conv2'),
        BatchNormalization(name='bn2'),
        MaxPooling2D(name='pool2'),
        Dropout(0.25, name='drop2'),

        # ── Convolutional Block 3 ────────────────────────────────
        Conv2D(128, (3,3), padding='same', activation='relu', name='conv3'),
        BatchNormalization(name='bn3'),
        MaxPooling2D(name='pool3'),
        Dropout(0.35, name='drop3'),

        # ── Fully Connected Layers (3 dense incl. output) ────────
        GlobalAveragePooling2D(name='gap'),
        Dense(256, activation='relu', name='fc1'),
        Dropout(0.45, name='drop_fc1'),
        Dense(128, activation='relu', name='fc2'),
        Dropout(0.25, name='drop_fc2'),
        Dense(n_classes, activation='softmax', name='output')   # FC3 = output
    ], name='Baseline_CNN')

    model.compile(
        optimizer=Adam(learning_rate=2e-4),  # Optimized learning rate
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

model_baseline = build_baseline_cnn()
model_baseline.summary()
print(f"\n  Total params : {model_baseline.count_params():,}")
print("  Architecture: 3 Conv2D layers + 3 Dense layers (incl. output) — matches spec exactly.")

Model: "Baseline_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1 (Conv2D)                  │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn1 (BatchNormalization)        │ (None, 48, 48, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling2D)            │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop1 (Dropout)                 │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn2 (BatchNormalization)        │ (None, 24, 24, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling2D)            │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop2 (Dropout)                 │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bn3 (BatchNormalization)        │ (None, 12, 12, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling2D)            │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop3 (Dropout)                 │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_fc1 (Dropout)              │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ drop_fc2 (Dropout)              │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,391 (626.53 KB)

 Trainable params: 159,943 (624.78 KB)

 Non-trainable params: 448 (1.75 KB)


  Total params : 160,391
  Architecture: 3 Conv2D layers + 3 Dense layers (incl. output) — matches spec exactly.


## Cell 9 — Train Baseline CNN

In [32]:
print("Training Baseline CNN …")

# Cosine LR schedule gives faster early learning and smoother late convergence.
def cosine_lr(epoch, total=EPOCHS, lr_max=2e-4, lr_min=8e-6):
    cos_inner = np.pi * (epoch / max(1, total - 1))
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(cos_inner))

callbacks_baseline = [
    EarlyStopping(monitor='val_loss', patience=12, 
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=5e-6, verbose=1),
    tf.keras.callbacks.LearningRateScheduler(lambda e: cosine_lr(e), verbose=0),
]

t0 = time.time()
hist_baseline = model_baseline.fit(
    train_aug_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_gen,
    validation_steps=val_steps,
    class_weight=class_weights,
    callbacks=callbacks_baseline,
    verbose=1
)


Training Baseline CNN …
Epoch 1/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 25s 22ms/step - accuracy: 0.2119 - loss: 1.9546 - val_accuracy: 0.2740 - val_loss: 1.7576 - learning_rate: 2.0000e-04
Epoch 2/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.2450 - loss: 1.8628 - val_accuracy: 0.2926 - val_loss: 1.7557 - learning_rate: 1.9980e-04
Epoch 3/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.2607 - loss: 1.8385 - val_accuracy: 0.3082 - val_loss: 1.7121 - learning_rate: 1.9921e-04
Epoch 4/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.2796 - loss: 1.8162 - val_accuracy: 0.3139 - val_loss: 1.6767 - learning_rate: 1.9823e-04
Epoch 5/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.3031 - loss: 1.7714 - val_accuracy: 0.3335 - val_loss: 1.8383 - learning_rate: 1.9686e-04
Epoch 6/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 13s 15ms/step - accuracy: 0.3191 - loss: 1.7335 - val_accuracy: 0.3311 - val_loss: 2.1383 - learning_rate: 1.9511e-04
Epoch 7/50
840/840 ━━━━━━━━━

## Cell 10 — Part A: Deeper CNN with BatchNorm & Dropout

The deeper architecture doubles the baseline in both layer count and filter depth:
- 8 Conv2D layers (vs 3 in baseline) across 4 blocks: 64→64→128→128→256→256→512→512
- BatchNormalization after each block and in FC layers
- Dropout rates increase with depth (0.20→0.30→0.40→0.50 in Conv blocks, 0.55/0.35 in FC)
- Compiled with Adam (lr=2e-4)

In [33]:

print("  PART A: DEEPER CNN WITH BATCHNORM & DROPOUT")



# MODEL BUILDING

def build_deep_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 1), n_classes=N_CLASSES):
    model = Sequential([
        # Block 1
        Conv2D(64, (3,3), padding='same', activation='relu', input_shape=input_shape),
        BatchNormalization(),
        Conv2D(64, (3,3), padding='same', activation='relu'),
        MaxPooling2D(),
        Dropout(0.20),

        # Block 2
        Conv2D(128, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(128, (3,3), padding='same', activation='relu'),
        MaxPooling2D(),
        Dropout(0.30),

        # Block 3
        Conv2D(256, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(256, (3,3), padding='same', activation='relu'),
        MaxPooling2D(),
        Dropout(0.40),

        # Block 4
        Conv2D(512, (3,3), padding='same', activation='relu'),
        BatchNormalization(),
        Conv2D(512, (3,3), padding='same', activation='relu'),
        Dropout(0.50),

        # Head
        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        BatchNormalization(),
        Dropout(0.55),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dropout(0.35),
        Dense(n_classes, activation='softmax')
    ])

    model.compile(
        optimizer=Adam(learning_rate=1.5e-4),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model



# MODEL INITIALIZATION

model_deep = build_deep_cnn()
model_deep.summary()

print(f"\n  Total params : {model_deep.count_params():,}")
print("  Deep CNN with BatchNorm + Dropout + GAP")



# CALLBACKS

callbacks_deep = [
    EarlyStopping(
        monitor='val_loss',
        patience=14,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        min_lr=3e-6,
        verbose=1
    )
]


# TRAINING

print("\nTraining Deep CNN …")

t0 = time.time()

hist_deep = model_deep.fit(
    train_aug_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_gen,
    validation_steps=val_steps,
    class_weight=class_weights,
    callbacks=callbacks_deep,
    verbose=1
)

deep_time = time.time() - t0


# OUTPUT SUMMARY

print("\n Deep CNN Training Completed")
print(f" Training Time : {deep_time:.2f} seconds")
print(f" Final Train Accuracy : {hist_deep.history['accuracy'][-1]:.4f}")
print(f" Final Val Accuracy   : {hist_deep.history['val_accuracy'][-1]:.4f}")

  PART A: DEEPER CNN WITH BATCHNORM & DROPOUT


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 6, 6, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 6, 6, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 6, 6, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 6, 6, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │             

 Total params: 5,086,919 (19.41 MB)

 Trainable params: 5,083,463 (19.39 MB)

 Non-trainable params: 3,456 (13.50 KB)


  Total params : 5,086,919
  Deep CNN with BatchNorm + Dropout + GAP

Training Deep CNN …
Epoch 1/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 36s 29ms/step - accuracy: 0.1647 - loss: 2.6501 - val_accuracy: 0.1968 - val_loss: 1.8758 - learning_rate: 1.5000e-04
Epoch 2/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.1979 - loss: 2.2314 - val_accuracy: 0.2776 - val_loss: 1.7916 - learning_rate: 1.5000e-04
Epoch 3/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.2107 - loss: 2.0922 - val_accuracy: 0.2635 - val_loss: 1.7607 - learning_rate: 1.5000e-04
Epoch 4/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.2543 - loss: 1.9392 - val_accuracy: 0.3153 - val_loss: 1.6578 - learning_rate: 1.5000e-04
Epoch 5/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.2725 - loss: 1.8804 - val_accuracy: 0.3570 - val_loss: 1.6537 - learning_rate: 1.5000e-04
Epoch 6/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - accuracy: 0.3318 - loss: 1.7745 - val_accuracy: 0.4594 - val_lo

## Cell 11 — Evaluate & Compare: Baseline vs Deep CNN

In [34]:
import time

start = time.time()
# train deep CNN model
deep_time = time.time() - start
print("  EVALUATION: BASELINE vs DEEP CNN")


res_base = evaluate_model(model_baseline, X_test, y_test, "Baseline CNN")
res_deep = evaluate_model(model_deep,     X_test, y_test, "Deep CNN")

# ── Training curves ───────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for col, (hist, label, color) in enumerate([
        (hist_baseline, 'Baseline CNN', '#e74c3c'),
        (hist_deep,     'Deep CNN',     '#3498db')]):
    axes[0,col].plot(hist.history['accuracy'],     label='Train', color=color, lw=2)
    axes[0,col].plot(hist.history['val_accuracy'], label='Val',   color=color, lw=2, linestyle='--')
    axes[0,col].set_title(f'{label} — Accuracy', fontsize=11, fontweight='bold')
    axes[0,col].set_xlabel('Epoch'); axes[0,col].set_ylabel('Accuracy')
    axes[0,col].legend(); axes[0,col].grid(alpha=0.3)

    axes[1,col].plot(hist.history['loss'],     label='Train', color=color, lw=2)
    axes[1,col].plot(hist.history['val_loss'], label='Val',   color=color, lw=2, linestyle='--')
    axes[1,col].set_title(f'{label} — Loss', fontsize=11, fontweight='bold')
    axes[1,col].set_xlabel('Epoch'); axes[1,col].set_ylabel('Loss')
    axes[1,col].legend(); axes[1,col].grid(alpha=0.3)

plt.suptitle('Training & Validation Curves — Baseline vs Deep CNN',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'training_curves_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("  Saved: training_curves_comparison.png")

# ── Confusion matrices ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, res in zip(axes, [res_base, res_deep]):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
                ax=ax, linewidths=0.4)
    ax.set_title(res['name'], fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted')
    ax.set_xticklabels(EMOTION_LABELS, rotation=30, ha='right')

plt.suptitle('Confusion Matrices — Baseline vs Deep CNN',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'confusion_matrices_AB.png', dpi=200, bbox_inches='tight')
plt.show()
print("  Saved: confusion_matrices_AB.png")

#Individual confusion matrices
for res in [res_base, res_deep]:
    cm = confusion_matrix(y_test, res['y_pred'])
    fig, ax = plt.subplots(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=EMOTION_LABELS, yticklabels=EMOTION_LABELS,
                ax=ax, linewidths=0.4)
    ax.set_title(f'Confusion Matrix - {res["name"]}', fontsize=11, fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted')
    ax.set_xticklabels(EMOTION_LABELS, rotation=30, ha='right')
    plt.tight_layout()
    fname = f'cm_{res["name"].replace(" ","_")}.png'
    plt.savefig(REPORT_DIR / fname, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"  Saved: {fname}")

#Summary table 
print("\nCOMPARISON TABLE:")
print(f"{'Model':<18} {'Accuracy':>10} {'Macro F1':>10} {'Train Time':>12}")
print("-" * 55)
print(f"  {'Baseline CNN':<16} {res_base['acc']:>10.4f} {res_base['f1']:>10.4f} {baseline_time:>10.1f}s")
print(f"  {'Deep CNN':<16} {res_deep['acc']:>10.4f} {res_deep['f1']:>10.4f} {deep_time:>10.1f}s")

# Discussion
print("\nOBSERVATIONS:")
if res_deep['acc'] > res_base['acc']:
    print("   Deeper CNN outperforms baseline — additional depth captures richer features")
else:
    print("   Deeper CNN did not improve over baseline — may indicate instability or overfitting")
    print("    Root cause: ImageDataGenerator + class_weight interaction with deeper architectures")


  EVALUATION: BASELINE vs DEEP CNN
53/53 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step

Baseline CNN Results:
Accuracy: 0.48622754491017967
Macro F1: 0.4171332181706022
              precision    recall  f1-score   support

           0       0.17      0.63      0.27       100
           1       0.78      0.18      0.29       100
           2       0.50      0.05      0.09       270
           3       0.64      0.78      0.70       300
           4       0.48      0.60      0.53       300
           5       0.47      0.25      0.33       300
           6       0.66      0.76      0.71       300

    accuracy                           0.49      1670
   macro avg       0.53      0.46      0.42      1670
weighted avg       0.54      0.49      0.46      1670

53/53 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step

Deep CNN Results:
Accuracy: 0.651497005988024
Macro F1: 0.6366484269575604
              precision    recall  f1-score   support

           0       0.31      0.50      0.38       100
           1       0.

In [36]:

print("COMPUTATIONAL EFFICIENCY ANALYSIS — PART A MODELS")


def model_complexity_metrics(model):
    total_params = model.count_params()

    trainable_params = np.sum([np.prod(w.shape) for w in model.trainable_weights])
    non_trainable_params = np.sum([np.prod(w.shape) for w in model.non_trainable_weights])

    model_size_mb = (total_params * 4) / (1024 * 1024)

    return {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'non_trainable_params': non_trainable_params,
        'model_size_mb': model_size_mb
    }


baseline_metrics = model_complexity_metrics(model_baseline)
deep_metrics = model_complexity_metrics(model_deep)

print("\nMODEL COMPLEXITY BREAKDOWN:")
print(f"{'Metric':<30} {'Baseline':<20} {'Deep CNN':<20}")
print("-"*70)

print(f"{'Total Parameters':<30} {baseline_metrics['total_params']:>18,} {deep_metrics['total_params']:>18,}")
print(f"{'Trainable Parameters':<30} {baseline_metrics['trainable_params']:>18,} {deep_metrics['trainable_params']:>18,}")
print(f"{'Model Size (MB)':<30} {baseline_metrics['model_size_mb']:>18.2f} {deep_metrics['model_size_mb']:>18.2f}")


params_increase = ((deep_metrics['total_params'] / baseline_metrics['total_params']) - 1) * 100
time_increase = ((deep_time / baseline_time) - 1) * 100
acc_improvement = (res_deep['acc'] - res_base['acc']) * 100

print(f"\nTRAINING EFFICIENCY METRICS:")
print(f"{'Metric':<30} {'Baseline':<20} {'Deep CNN':<20} {'Change':<15}")
print("-"*80)

print(f"{'Training Time (s)':<30} {baseline_time:>18.2f} {deep_time:>18.2f} {time_increase:>13.1f}%")
print(f"{'Parameters (Millions)':<30} {baseline_metrics['total_params']/1e6:>18.2f} {deep_metrics['total_params']/1e6:>18.2f} {params_increase:>13.1f}%")
print(f"{'Accuracy':<30} {res_base['acc']:>18.4f} {res_deep['acc']:>18.4f} +{acc_improvement:>12.1f}%")

efficiency_baseline = res_base['acc'] / baseline_time if baseline_time > 0 else 0
efficiency_deep = res_deep['acc'] / deep_time if deep_time > 0 else 0

print(f"\nEFFICIENCY RATIO (Accuracy / Training Time):")
print(f"  Baseline CNN: {efficiency_baseline:.6f} acc/sec")
print(f"  Deep CNN    : {efficiency_deep:.6f} acc/sec")
print(f"  Efficiency change: {(efficiency_deep / efficiency_baseline):.2f}x")


# FINAL ANALYSIS
print("\nFINAL ANALYSIS:")
print(f"  Accuracy improvement: +{acc_improvement:.2f}%")
print(f"  Computational cost increase: +{time_increase:.1f}%")
print("  Deep CNN improves feature learning but increases computational cost.")
print("  Final model choice depends on trade-off between accuracy and efficiency.")

COMPUTATIONAL EFFICIENCY ANALYSIS — PART A MODELS

MODEL COMPLEXITY BREAKDOWN:
Metric                         Baseline             Deep CNN            
----------------------------------------------------------------------
Total Parameters                          160,391          5,086,919
Trainable Parameters                      159,943          5,083,463
Model Size (MB)                              0.61              19.41

TRAINING EFFICIENCY METRICS:
Metric                         Baseline             Deep CNN             Change         
--------------------------------------------------------------------------------
Training Time (s)                            0.00               0.00           0.0%
Parameters (Millions)                        0.16               5.09        3071.6%
Accuracy                                   0.4862             0.6515 +        16.5%

EFFICIENCY RATIO (Accuracy / Training Time):
  Baseline CNN: 14463.731465 acc/sec
  Deep CNN    : 19379.975165 acc/se

## Cell 12 — Optimiser Analysis: SGD vs Adam

In [38]:

print("=" * 60)
print("  CELL 12 — OPTIMISER ANALYSIS: SGD vs Adam")
print("=" * 60)


# Build SGD model
model_sgd = build_deep_cnn()
model_sgd.compile(
    optimizer=SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_sgd._name = 'Deep_CNN_SGD'

print("Training Deep CNN with SGD (lr=0.01, momentum=0.9, Nesterov=True) …")

t0 = time.time()

hist_sgd = model_sgd.fit(
    train_aug_gen,
    steps_per_epoch=steps_per_epoch,
    epochs=min(EPOCHS, 15),
    validation_data=val_gen,
    validation_steps=val_steps,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=6,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-6, verbose=0)
    ],
    verbose=1
)

sgd_time = time.time() - t0
print(f"  SGD training done in {sgd_time:.1f}s")


# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lbl, hist, color in [
    ('Adam', hist_deep, '#3498db'),
    ('SGD', hist_sgd, '#e74c3c')
]:
    axes[0].plot(hist.history['val_accuracy'], label=lbl, color=color, lw=2)
    axes[1].plot(hist.history['val_loss'], label=lbl, color=color, lw=2)

axes[0].set_title('Validation Accuracy — Adam vs SGD')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Validation Loss — Adam vs SGD')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'optimizer_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nOptimizer Comparison Complete")
print("Saved: optimizer_comparison.png")


adam_best = max(hist_deep.history['val_accuracy'])
sgd_best = max(hist_sgd.history['val_accuracy'])

print(f"\nBest Validation Accuracy:")
print(f"  Adam: {adam_best:.4f}")
print(f"  SGD : {sgd_best:.4f}")

print("\nFINAL ANALYSIS:")
if adam_best >= sgd_best:
    print("  Adam performed better due to adaptive learning rates.")
else:
    print("  SGD performed competitively but requires more tuning and epochs.")

  CELL 12 — OPTIMISER ANALYSIS: SGD vs Adam
Training Deep CNN with SGD (lr=0.01, momentum=0.9, Nesterov=True) …
Epoch 1/15
840/840 ━━━━━━━━━━━━━━━━━━━━ 32s 27ms/step - accuracy: 0.1907 - loss: 2.2476 - val_accuracy: 0.2073 - val_loss: 1.8848 - learning_rate: 0.0100
Epoch 2/15
840/840 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.2034 - loss: 2.0197 - val_accuracy: 0.2495 - val_loss: 1.8182 - learning_rate: 0.0100
Epoch 3/15
840/840 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.2143 - loss: 1.9692 - val_accuracy: 0.2534 - val_loss: 1.8152 - learning_rate: 0.0100
Epoch 4/15
840/840 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.2253 - loss: 1.9357 - val_accuracy: 0.2563 - val_loss: 1.7795 - learning_rate: 0.0100
Epoch 5/15
840/840 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.2306 - loss: 1.9083 - val_accuracy: 0.1960 - val_loss: 1.9491 - learning_rate: 0.0100
Epoch 6/15
840/840 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - accuracy: 0.2533 - loss: 1.8604 - val_accuracy: 0.3025 - val_l

## Cell 13 — Ablation Study: Effect of Dropout & BatchNorm

In [44]:
print(" DROPOUT & BATCHNORM ABLATION STUDY")

import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)

def train_ablation(model):
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=0
    )

    history = model.fit(
        train_aug_gen,
        steps_per_epoch=steps_per_epoch,
        epochs=15,
        validation_data=val_gen,
        validation_steps=val_steps,
        class_weight=class_weights,
        callbacks=[early_stop],
        verbose=0
    )

    return {
        "val_acc": max(history.history["val_accuracy"]),
        "val_loss": min(history.history["val_loss"])
    }


ablation_configs = [
    ("No Dropout", False, True),
    ("No BatchNorm", True, False),
    ("No Both", False, False),
]

ablation_results = {}

for label, use_dropout, use_bn in ablation_configs:
    print(f"Training: {label}")

    # IMPORTANT: clear old model memory (prevents slowdown/crashes)
    tf.keras.backend.clear_session()

    model = build_ablation_cnn(use_dropout, use_bn)
    result = train_ablation(model)

    ablation_results[label] = result

    print(f"  Best val accuracy: {result['val_acc']:.4f}")


# Full model comparison 
ablation_results["Full Model"] = {
    "val_acc": max(hist_deep.history["val_accuracy"]),
    "val_loss": min(hist_deep.history["val_loss"])
}

print("\n FINAL RESULTS:")
for name, res in ablation_results.items():
    print(f"{name:<15}: Accuracy = {res['val_acc']:.4f}, Loss = {res['val_loss']:.4f}")

 DROPOUT & BATCHNORM ABLATION STUDY
Training: No Dropout
  Best val accuracy: 0.5370
Training: No BatchNorm
  Best val accuracy: 0.5095
Training: No Both
  Best val accuracy: 0.5468

 FINAL RESULTS:
No Dropout     : Accuracy = 0.5370, Loss = 1.2313
No BatchNorm   : Accuracy = 0.5095, Loss = 1.2908
No Both        : Accuracy = 0.5468, Loss = 1.2205
Full Model     : Accuracy = 0.6759, Loss = 0.8799


## Cell 14 — Part B: Transfer Learning with MobileNetV2

MobileNetV2 (pre-trained on ImageNet) is adapted using a two-phase strategy:
- **Phase 1 — Feature Extraction**: Freeze entire base, train only custom head (20 epochs, lr=5×10⁻⁴)
- **Phase 2 — Fine-Tuning**: Unfreeze top 40 layers, retrain at lower lr=5×10⁻⁵ (25 epochs)

Input: grayscale images converted to RGB 96×96 using `preprocess_input` for correct normalisation.

In [45]:
print("=" * 60)
print("  CELL 14 — PART B: TRANSFER LEARNING — MobileNetV2")
print("=" * 60)

import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# ── Phase 1: Feature Extraction ───────────────────────────────
print("Phase 1 — Feature Extraction (Frozen Base)")

base = MobileNetV2(
    input_shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3),
    include_top=False,
    weights='imagenet'
)

base.trainable = False

inputs = Input(shape=(IMG_SIZE_TL, IMG_SIZE_TL, 3))
x = base(inputs, training=False)
x = GlobalAveragePooling2D()(x)

x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

outputs = Dense(N_CLASSES, activation='softmax')(x)

model_mobilenet = Model(inputs, outputs)

model_mobilenet.compile(
    optimizer=Adam(learning_rate=4e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"  Trainable params (Phase 1): {model_mobilenet.count_params():,}")

cb_tl = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=0),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=0)
]

t0 = time.time()

hist_tl_phase1 = model_mobilenet.fit(
    X_tr_rgb, y_train,
    sample_weight=np.array([class_weights[y] for y in y_train]),
    batch_size=BATCH_SIZE,
    epochs=50,
    validation_data=(X_val_rgb, y_val),
    callbacks=cb_tl,
    verbose=1
)

# ── Phase 2: Fine-Tuning ─────────────────────────────────────
print("\nPhase 2 — Fine-Tuning (Unfreezing Base Model)")

base.trainable = True

model_mobilenet.compile(
    optimizer=Adam(learning_rate=3e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"  Trainable params (Phase 2): {model_mobilenet.count_params():,}")

hist_tl_phase2 = model_mobilenet.fit(
    X_tr_rgb, y_train,
    sample_weight=np.array([class_weights[y] for y in y_train]),
    batch_size=16,
    epochs=EPOCHS_FINE,
    validation_data=(X_val_rgb, y_val),
    callbacks=cb_tl,
    verbose=1
)

mobilenet_time = time.time() - t0

print(f"\nMobileNetV2 complete: {mobilenet_time:.1f}s ({mobilenet_time/60:.1f} min)")

# ── Merge histories safely ────────────────────────────────────
class MergedHistory:
    def __init__(self, h1, h2):
        self.history = {}

        for k in set(h1.history.keys()).union(h2.history.keys()):
            self.history[k] = h1.history.get(k, []) + h2.history.get(k, [])

hist_mobilenet = MergedHistory(hist_tl_phase1, hist_tl_phase2)
phase1_len = len(hist_tl_phase1.history['accuracy'])

  CELL 14 — PART B: TRANSFER LEARNING — MobileNetV2
Phase 1 — Feature Extraction (Frozen Base)
  Trainable params (Phase 1): 3,050,055
Epoch 1/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 32s 26ms/step - accuracy: 0.2991 - loss: 2.2492 - val_accuracy: 0.4643 - val_loss: 1.4204 - learning_rate: 4.0000e-04
Epoch 2/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.4052 - loss: 1.6464 - val_accuracy: 0.4761 - val_loss: 1.3689 - learning_rate: 4.0000e-04
Epoch 3/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.4511 - loss: 1.4972 - val_accuracy: 0.4920 - val_loss: 1.3192 - learning_rate: 4.0000e-04
Epoch 4/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.4662 - loss: 1.4277 - val_accuracy: 0.4907 - val_loss: 1.3273 - learning_rate: 4.0000e-04
Epoch 5/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.4801 - loss: 1.3909 - val_accuracy: 0.5025 - val_loss: 1.3000 - learning_rate: 4.0000e-04
Epoch 6/50
840/840 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.4962 - loss:

2026-05-10 23:07:05.566621: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-10 23:07:05.749132: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-10 23:07:06.013906: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-10 23:07:06.218981: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1678/1680 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.3198 - loss: 2.1554

2026-05-10 23:08:04.897423: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-05-10 23:08:05.079807: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1680/1680 ━━━━━━━━━━━━━━━━━━━━ 123s 43ms/step - accuracy: 0.3199 - loss: 2.1549 - val_accuracy: 0.3600 - val_loss: 1.6037 - learning_rate: 3.0000e-05
Epoch 2/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 36s 22ms/step - accuracy: 0.4757 - loss: 1.3822 - val_accuracy: 0.4139 - val_loss: 1.5057 - learning_rate: 3.0000e-05
Epoch 3/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - accuracy: 0.5512 - loss: 1.1516 - val_accuracy: 0.4752 - val_loss: 1.3745 - learning_rate: 3.0000e-05
Epoch 4/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - accuracy: 0.6315 - loss: 0.9655 - val_accuracy: 0.5172 - val_loss: 1.3173 - learning_rate: 3.0000e-05
Epoch 5/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - accuracy: 0.6977 - loss: 0.7886 - val_accuracy: 0.5239 - val_loss: 1.3542 - learning_rate: 3.0000e-05
Epoch 6/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms/step - accuracy: 0.7803 - loss: 0.6029 - val_accuracy: 0.5233 - val_loss: 1.4564 - learning_rate: 3.0000e-05
Epoch 7/50
1680/1680 ━━━━━━━━━━━━━━━━━━━━ 37s 22ms

## Cell 15 — Evaluate MobileNetV2 & Full Model Comparison

In [46]:
print("=" * 60)
print("  CELL 15 — PART B: EVALUATE MobileNetV2 + FULL COMPARISON")
print("=" * 60)

# ── MobileNetV2 Evaluation ───────────────────────────────────
res_mobilenet = evaluate_model(
    model_mobilenet,
    X_test_rgb,
    y_test,
    "MobileNetV2 (Transfer Learning)"
)

# ── Training curves ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_mobilenet.history['accuracy'], label='Train', lw=2)
axes[0].plot(hist_mobilenet.history['val_accuracy'], label='Val', lw=2, linestyle='--')
axes[0].axvline(x=phase1_len, color='gray', linestyle=':', label='Fine-tune start')
axes[0].set_title('MobileNetV2 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(hist_mobilenet.history['loss'], label='Train', lw=2)
axes[1].plot(hist_mobilenet.history['val_loss'], label='Val', lw=2, linestyle='--')
axes[1].axvline(x=phase1_len, color='gray', linestyle=':', label='Fine-tune start')
axes[1].set_title('MobileNetV2 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'mobilenet_training.png', dpi=200, bbox_inches='tight')
plt.show()

print("Saved: mobilenet_training.png")

# ── Confusion Matrix ─────────────────────────────────────────
cm_mn = confusion_matrix(y_test, res_mobilenet['y_pred'])

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_mn,
    annot=True,
    fmt='d',
    cmap='Purples',
    xticklabels=EMOTION_LABELS,
    yticklabels=EMOTION_LABELS,
    ax=ax
)

ax.set_title('Confusion Matrix — MobileNetV2')
ax.set_xlabel('Predicted')
ax.set_ylabel('True Label')

plt.tight_layout()
plt.savefig(REPORT_DIR / 'confusion_matrix_mobilenet.png', dpi=200, bbox_inches='tight')
plt.show()

print("Saved: confusion_matrix_mobilenet.png")

# ── SAFE TIME HANDLING (FIXED BUG) ────────────────────────────
if 'baseline_time' not in globals():
    baseline_time = 0
if 'deep_time' not in globals():
    deep_time = 0
if 'mobilenet_time' not in globals():
    mobilenet_time = 0

# ── FULL MODEL COMPARISON ─────────────────────────────────────
all_results = [res_base, res_deep, res_mobilenet]
all_names = [r['name'] for r in all_results]
all_times = [baseline_time, deep_time, mobilenet_time]

x = np.arange(len(all_names))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bars1 = axes[0].bar(x - w/2, [r['acc'] for r in all_results], w, label='Accuracy')
bars2 = axes[0].bar(x + w/2, [r['f1'] for r in all_results], w, label='Macro F1')

axes[0].set_ylim(0, 1.05)
axes[0].set_xticks(x)
axes[0].set_xticklabels(all_names, rotation=15, ha='right')
axes[0].set_title('Model Performance Comparison')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

for b in list(bars1) + list(bars2):
    axes[0].text(
        b.get_x() + b.get_width()/2,
        b.get_height() + 0.005,
        f'{b.get_height():.3f}',
        ha='center',
        fontsize=8
    )

axes[1].bar(all_names, all_times)
axes[1].set_title('Training Time Comparison')
axes[1].set_ylabel('Seconds')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(REPORT_DIR / 'full_model_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

print("Saved: full_model_comparison.png")

# ── FINAL TABLE ───────────────────────────────────────────────
print("\n" + "="*65)
print("FINAL MODEL COMPARISON")
print("="*65)
print(f"{'Model':<30} {'Accuracy':>10} {'Macro F1':>10} {'Time':>10}")
print("-"*65)

for r, t in zip(all_results, all_times):
    print(f"{r['name']:<30} {r['acc']:>10.4f} {r['f1']:>10.4f} {t:>10.1f}s")

best_model = max(all_results, key=lambda x: x['acc'])

print("\nBEST MODEL:")
print(f"  {best_model['name']} with Accuracy = {best_model['acc']:.4f}")

# ── SAVE RESULTS ──────────────────────────────────────────────
summary = {
    "Baseline": res_base['acc'],
    "DeepCNN": res_deep['acc'],
    "MobileNetV2": res_mobilenet['acc'],
    "Best_Model": best_model['name']
}

with open(REPORT_DIR / "results_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved: results_summary.json")

  CELL 15 — PART B: EVALUATE MobileNetV2 + FULL COMPARISON
53/53 ━━━━━━━━━━━━━━━━━━━━ 10s 119ms/step

MobileNetV2 (Transfer Learning) Results:
Accuracy: 0.48562874251497007
Macro F1: 0.47077620255172054
              precision    recall  f1-score   support

           0       0.18      0.46      0.26       100
           1       0.71      0.47      0.57       100
           2       0.35      0.21      0.26       270
           3       0.65      0.67      0.66       300
           4       0.52      0.39      0.45       300
           5       0.38      0.44      0.41       300
           6       0.67      0.70      0.69       300

    accuracy                           0.49      1670
   macro avg       0.50      0.48      0.47      1670
weighted avg       0.51      0.49      0.49      1670

Saved: mobilenet_training.png
Saved: confusion_matrix_mobilenet.png
Saved: full_model_comparison.png

FINAL MODEL COMPARISON
Model                            Accuracy   Macro F1       Time
-----------

## Cell 16 — Sample Predictions & Error Analysis

In [53]:
print("=" * 60)
print("  CELL 16 — SAMPLE PREDICTIONS & ERROR ANALYSIS")
print("=" * 60)

# ── Safe fallback for BEST_MODEL ─────────────────────────────
if 'BEST_MODEL' not in globals():
    BEST_MODEL = model_mobilenet
    BEST_MODEL_NAME = 'MobileNetV2 (fallback)'
    BEST_MODEL_INPUT = 'rgb'

X_eval = X_test if BEST_MODEL_INPUT == 'gray' else X_test_rgb
probs = BEST_MODEL.predict(X_eval, verbose=0)
preds = probs.argmax(axis=1)

print(f"Using inference model: {BEST_MODEL_NAME} | input={BEST_MODEL_INPUT}")

# ── Correct vs Incorrect samples ─────────────────────────────
correct_idx = np.where(preds == y_test)[0][:3]
incorrect_idx = np.where(preds != y_test)[0][:3]

fig, axes = plt.subplots(2, 3, figsize=(12, 8))

for row, idxs in enumerate([correct_idx, incorrect_idx]):
    for col, ix in enumerate(idxs):
        axes[row, col].imshow(X_test[ix].squeeze(), cmap='gray', vmin=0, vmax=1)

        true_lbl = EMOTION_LABELS[y_test[ix]]
        pred_lbl = EMOTION_LABELS[preds[ix]]
        conf = probs[ix][preds[ix]]

        color = '#27ae60' if preds[ix] == y_test[ix] else '#e74c3c'

        axes[row, col].set_title(
            f"True: {true_lbl}\nPred: {pred_lbl} ({conf*100:.1f}%)",
            fontsize=9,
            color=color,
            fontweight='bold'
        )
        axes[row, col].axis('off')

plt.suptitle('Sample Predictions on Test Images', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'sample_predictions.png', dpi=200, bbox_inches='tight')
plt.show()

print("  Saved: sample_predictions.png")

# ── Error analysis ───────────────────────────────────────────
print("\nERROR ANALYSIS — Misclassified Examples:")

for ix in incorrect_idx:
    true_lbl = EMOTION_LABELS[y_test[ix]]
    pred_lbl = EMOTION_LABELS[preds[ix]]
    conf = probs[ix][preds[ix]]

    top3_idx = probs[ix].argsort()[::-1][:3]

    print(f"  True: {true_lbl:<10} → Pred: {pred_lbl:<10} ({conf*100:.1f}%)")
    print(f"    Top-3: {[(EMOTION_LABELS[i], f'{probs[ix][i]*100:.1f}%') for i in top3_idx]}")

print("\nPotential reasons for misclassification:")
print("  1. High inter-class similarity (Fear↔Surprise, Sad↔Neutral, Angry↔Disgust)")
print("  2. Low-resolution images reduce feature detail")
print("  3. Class imbalance affects learning bias")
print("  4. Ambiguous facial expressions even for humans")

  CELL 16 — SAMPLE PREDICTIONS & ERROR ANALYSIS
Using inference model: MobileNetV2 (fallback) | input=rgb
  Saved: sample_predictions.png

ERROR ANALYSIS — Misclassified Examples:
  True: Angry      → Pred: Happy      (39.6%)
    Top-3: [('Happy', '39.6%'), ('Sad', '30.0%'), ('Angry', '17.8%')]
  True: Angry      → Pred: Happy      (73.5%)
    Top-3: [('Happy', '73.5%'), ('Angry', '14.8%'), ('Sad', '5.7%')]
  True: Angry      → Pred: Neutral    (25.2%)
    Top-3: [('Neutral', '25.2%'), ('Angry', '22.7%'), ('Disgust', '14.6%')]

Potential reasons for misclassification:
  1. High inter-class similarity (Fear↔Surprise, Sad↔Neutral, Angry↔Disgust)
  2. Low-resolution images reduce feature detail
  3. Class imbalance affects learning bias
  4. Ambiguous facial expressions even for humans


## Cell 17 — Save Trained Models

In [62]:
print("=" * 60)
print("  CELL 17 — SAVE TRAINED MODELS")
print("=" * 60)

models_dir = Path('models')
models_dir.mkdir(parents=True, exist_ok=True)

for model, fname in [
    (model_baseline, 'model_baseline.keras'),
    (model_deep,     'model_deep.keras'),
    (model_mobilenet,'model_mobilenet.keras')
]:
    try:
        model.save(models_dir / fname)
        print(f"  Saved: {fname}")
    except Exception as e:
        print(f"  Could not save {fname}: {e}")

# Smoke test — quick inference check
print("\nSMOKE TEST:")
for name, mdl, X_in in [
    ('Baseline CNN', model_baseline, X_test[:1]),
    ('Deep CNN',     model_deep,     X_test[:1]),
    ('MobileNetV2',  model_mobilenet, X_test_rgb[:1])
]:
    try:
        p = mdl.predict(X_in, verbose=0)
        print(f"  {name}: shape={p.shape}  top={EMOTION_LABELS[p.argmax()]}  conf={p.max():.3f}")
    except Exception as e:
        print(f"  {name}: FAILED — {e}")


  CELL 17 — SAVE TRAINED MODELS
  Saved: model_baseline.keras
  Saved: model_deep.keras
  Saved: model_mobilenet.keras

SMOKE TEST:
  Baseline CNN: shape=(1, 7)  top=Angry  conf=0.676
  Deep CNN: shape=(1, 7)  top=Angry  conf=0.931
  MobileNetV2: shape=(1, 7)  top=Angry  conf=0.924


## Cell 18 — GRADIO

In [63]:
print("=" * 60)
print("  CELL 17 — SAVE MODELS + GRADIO DEMO")
print("=" * 60)

from pathlib import Path
import numpy as np
import gradio as gr

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)

# ============================================================
# SAVE MODELS (FIXED)
# ============================================================

model_list = [
    (model_baseline, "model_baseline.keras"),
    (model_deep, "model_deep.keras"),
    (model_mobilenet, "model_mobilenet.keras")
]

for model, fname in model_list:
    model.save(models_dir / fname)
    print(f"Saved: {fname}")

# ============================================================
# SMOKE TEST
# ============================================================

print("\nSMOKE TEST:")

for model, fname in model_list:
    X_in = X_test_rgb[:1] if "mobilenet" in fname else X_test[:1]

    preds = model.predict(X_in, verbose=0)
    label = EMOTION_LABELS[np.argmax(preds)]

    print(f"{fname:<25} → {label} | conf={np.max(preds):.3f}")

# ============================================================
# GRADIO FUNCTION
# ============================================================

def predict_emotion(img):

    img = np.array(img)

    # grayscale for CNN models (baseline/deep)
    if len(img.shape) == 3 and img.shape[-1] == 3:
        img_gray = np.mean(img, axis=-1)
        img_gray = img_gray / 255.0
        img_gray = img_gray.reshape(1, IMG_SIZE, IMG_SIZE, 1)

        preds = model_deep.predict(img_gray, verbose=0)

    else:
        img = img / 255.0
        img = img.reshape(1, IMG_SIZE_TL, IMG_SIZE_TL, 3)

        preds = model_mobilenet.predict(img, verbose=0)

    label = EMOTION_LABELS[np.argmax(preds)]
    confidence = float(np.max(preds))

    return f"{label} ({confidence:.2f})"

# ============================================================
# GRADIO UI
# ============================================================

demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(type="numpy", label="Upload Face Image"),
    outputs=gr.Textbox(label="Predicted Emotion"),
    title="Emotion Detection (CNN + MobileNetV2)",
    description="Upload a face image and get emotion prediction"
)

demo.launch()

  CELL 17 — SAVE MODELS + GRADIO DEMO
Saved: model_baseline.keras
Saved: model_deep.keras
Saved: model_mobilenet.keras

SMOKE TEST:
model_baseline.keras      → Angry | conf=0.676
model_deep.keras          → Angry | conf=0.931
model_mobilenet.keras     → Angry | conf=0.924
* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://24696cae3646e3da10.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created dataset file at: .gradio/flagged/dataset1.csv


## Cell 19 — Interactive GUI (Real-Time Prediction)

An ipywidgets-based GUI allowing users to upload any face image and get real-time emotion predictions from all three trained models plus a weighted ensemble. Confidence threshold of 65% flags uncertain predictions.

## Extended Analysis: Multiple Transfer Learning Models

This section evaluates and compares multiple pre-trained architectures to identify the best transfer learning approach for facial expression classification.


In [ ]:
print("=" * 80)
print("  EXTENDED ANALYSIS — MULTI-MODEL TRANSFER LEARNING COMPARISON")
print("=" * 80)

from tensorflow.keras.applications import ResNet50, VGG16, MobileNetV2
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path
import numpy as np
import time
import matplotlib.pyplot as plt
from PIL import Image

# ── CONFIG ───────────────────────────────────────────────
tl_configs = {
    'MobileNetV2_96': {
        'base_fn': lambda: MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet'),
        'input_size': 96,
        'freeze_until_layer': -40,
        'phase1_epochs': 5,
        'phase2_epochs': 5,
        'phase1_lr': 4e-4,
        'phase2_lr': 3e-5,
    },
    'ResNet50_224': {
        'base_fn': lambda: ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet'),
        'input_size': 224,
        'freeze_until_layer': 100,
        'phase1_epochs': 5,
        'phase2_epochs': 5,
        'phase1_lr': 5e-4,
        'phase2_lr': 2e-5,
    }
}

# ── USE EXISTING DATA  ────────────────────────────
X_train_rgb = X_tr_rgb
X_val_rgb   = X_val_rgb
X_test_rgb  = X_test_rgb

# ── RESIZE FUNCTION ───────────────────────────────────────
def prepare_data(X_arr, size):
    out = []
    for img in X_arr:
        pil = Image.fromarray((img * 255).astype('uint8'))
        pil = pil.convert("RGB").resize((size, size))
        arr = preprocess_input(np.array(pil, dtype=np.float32))
        out.append(arr)
    return np.array(out)

# ── PREPARE DATA ─────────────────────────────────────────
data_by_size = {}

for name, cfg in tl_configs.items():
    size = cfg['input_size']
    
    if size not in data_by_size:
        print(f"Preparing {size}x{size} data...")
        data_by_size[size] = {
            'train': prepare_data(X_train_rgb, size),
            'val': prepare_data(X_val_rgb, size),
            'test': prepare_data(X_test_rgb, size),
        }

# ── TRAIN MODELS ─────────────────────────────────────────
tl_results = {}

for name, cfg in tl_configs.items():
    print("\n" + "="*60)
    print("TRAINING:", name)
    
    size = cfg['input_size']
    X_tr = data_by_size[size]['train']
    X_va = data_by_size[size]['val']
    X_te = data_by_size[size]['test']
    
    base = cfg['base_fn']()
    base.trainable = False

    inputs = Input(shape=(size, size, 3))
    x = base(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    outputs = Dense(N_CLASSES, activation='softmax')(x)

    model = Model(inputs, outputs)

    model.compile(
        optimizer=Adam(cfg['phase1_lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    cb = [
        EarlyStopping(patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2)
    ]

    # Phase 1
    model.fit(X_tr, y_train,
              validation_data=(X_va, y_val),
              epochs=cfg['phase1_epochs'],
              batch_size=32,
              verbose=1,
              callbacks=cb)

    # Fine-tuning
    base.trainable = True
    model.compile(
        optimizer=Adam(cfg['phase2_lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    model.fit(X_tr, y_train,
              validation_data=(X_va, y_val),
              epochs=cfg['phase2_epochs'],
              batch_size=16,
              verbose=1,
              callbacks=cb)

    # Evaluation
    preds = model.predict(X_te).argmax(axis=1)

    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='macro')

    tl_results[name] = {
        "accuracy": acc,
        "f1": f1
    }

    print(f"{name} → Acc: {acc:.4f}, F1: {f1:.4f}")

# ── FINAL SUMMARY ─────────────────────────────────────────
print("\nFINAL RESULTS")
for k, v in tl_results.items():
    print(f"{k:<20} Accuracy: {v['accuracy']:.4f} | F1: {v['f1']:.4f}")

  EXTENDED ANALYSIS — MULTI-MODEL TRANSFER LEARNING COMPARISON
Preparing 96x96 data...
Preparing 224x224 data...
